# SARSA on FrozenLake

Tabular **on-policy** SARSA.

In [ ]:
import gymnasium as gym
import numpy as np
from matplotlib import pyplot as plt
import time

## Configuration

In [ ]:
# Training configuration
NUM_EPISODES = 5000
MAX_STEPS_PER_EPISODE = 100
LEARNING_RATE = 0.1          
DISCOUNT_FACTOR = 0.9
EPSILON_MAX = 1.0
EPSILON_MIN = 0.01           # set to 0.0 for convergence to the optimal policy
EXPLORATION_RATE_DECAY = 0.001
SEED = 42

In [ ]:
ACTION_SYMBOLS = np.array(["\u2190", "\u2193", "\u2192", "\u2191"])  # left, down, right, up

## Environment

In [ ]:
env = gym.make("FrozenLake-v1",
               map_name="4x4",  # 8x8
               render_mode="ansi",
               reward_schedule=(1, -1, 0),  # (goal, hole, step)
               is_slippery=False,
               # success_rate=3./4
               max_episode_steps=MAX_STEPS_PER_EPISODE,
               )

# Seed
env.action_space.seed(SEED)
env.observation_space.seed(SEED)
rng = np.random.default_rng(SEED)

In [ ]:
action_space_size = env.action_space.n
state_space_size = env.observation_space.n
print("state", state_space_size, "action", action_space_size)

q_table = np.zeros((state_space_size, action_space_size))

## Policy helpers

In [ ]:
def argmax_random_tiebreak(values, rng):
    """argmax that breaks ties uniformly at random instead of by lowest index."""
    best = np.flatnonzero(values == values.max())
    return int(rng.choice(best))


def epsilon_greedy(q_table, state, epsilon, rng):
    """Select an action using an epsilon-greedy policy."""
    if rng.uniform(0, 1) > epsilon:
        # Exploit: best action for this state
        return argmax_random_tiebreak(q_table[state, :], rng)
    else:
        # Explore: uniformly random action
        return env.action_space.sample()

## Training

In [ ]:
rewards_all_episodes = []
total_steps = []

epsilon = EPSILON_MAX

t0 = time.time()
for episode in range(NUM_EPISODES):
    # Initialize new episode
    state, info = env.reset() 
    rewards_current_episode = 0.0
    steps_taken = 0

    # Choose the FIRST action before the loop (on-policy)
    action = epsilon_greedy(q_table, state, epsilon, rng)

    for step in range(MAX_STEPS_PER_EPISODE):
        # Take action, observe next state and reward
        next_state, reward, terminated, truncated, info = env.step(action)

        # TD target for SARSA: Q(s, a) <- Q(s, a) + alpha * (reward + gamma * Q(s', a') - Q(s, a))
        if terminated:
            # The episode is over (no s' to act from, so there is no a' either)
            next_action = None # Do not modify this line!
            target = reward
        else:
            # SARSA: a' drawn from the epsilon-greedy policy
            next_action = epsilon_greedy(q_table, next_state, epsilon, rng)
            target = reward + DISCOUNT_FACTOR * q_table[next_state, next_action]

        # On-policy TD update for Q(s, a)
        q_table[state, action] = None # <-- Complete this line to update the Q-table using the SARSA update rule

        # Transition to next state AND next action
        state = next_state
        action = next_action

        rewards_current_episode += reward
        steps_taken = step + 1 

        if terminated or truncated:
            break

    total_steps.append(steps_taken)
    rewards_all_episodes.append(rewards_current_episode)

    # Exponential exploration-rate decay
    epsilon = EPSILON_MIN + (EPSILON_MAX - EPSILON_MIN) * np.exp(-EXPLORATION_RATE_DECAY * episode)

    if episode % 1000 == 0:
        print(f"Episode {episode + 1}: Total Reward: {rewards_current_episode}, Epsilon: {epsilon:.4f}")

print(f"Training time: {(time.time() - t0) / 60:.4f} [min]")

## Learning curves

In [ ]:
plt.figure(figsize=(8, 5))
window = min(1000, max(1, len(rewards_all_episodes) // 10))
average_rewards = np.convolve(np.array(rewards_all_episodes), np.ones(window) / window, mode="valid")
plt.plot(average_rewards)
plt.title(f"Rolling Average Reward (window = {window} episodes)")
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(total_steps)
plt.title("Number of steps per episode")
plt.xlabel("Episode")
plt.ylabel("steps")
plt.grid(True)
plt.tight_layout()
plt.show()

## Learned greedy policy

Terminal states (holes `H`, goal `G`) are masked out: their Q-rows are never updated,
because the episode ends the moment they are entered, so the arrow shown there would be
meaningless.

In [ ]:
desc = env.unwrapped.desc.astype(str).ravel()
HOLE_STATES = np.flatnonzero(desc == "H")
GOAL_STATE  = int(np.flatnonzero(desc == "G")[0])
n = env.unwrapped.desc.shape[0]     # 4 or 8

greedy_actions = np.array([argmax_random_tiebreak(q_table[s, :], rng)
                           for s in range(state_space_size)])

symbols = ACTION_SYMBOLS[greedy_actions].copy()
for s in HOLE_STATES:
    symbols[s] = "H"
symbols[GOAL_STATE] = "G"

print("Greedy policy:")
print(symbols.reshape(n, n))
print()
print("Q-table:")
print(np.round(q_table, 3))

## Roll out the greedy policy

In [ ]:
state, info = env.reset(seed=SEED)
print(env.render())

for _ in range(MAX_STEPS_PER_EPISODE):
    action = argmax_random_tiebreak(q_table[state, :], rng)
    state, reward, terminated, truncated, info = env.step(action)
    print(env.render())
    if terminated or truncated:
        print(f"Episode finished. reward={reward}, terminated={terminated}, truncated={truncated}")
        break

env.close()

## Save q-values

In [ ]:
np.save("table_sarsa.npy", q_table)
print("saved table_sarsa.npy", q_table.shape)